## Retrieval-Augmented Generation (RAG)

#### PART:1 langchain Document Loading

#### Instantiate PyPDFLoader with the path to your pdf

In [1]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = 'documents/sample.pdf'

loader = PyPDFLoader(PDF_PATH)

pages = loader.load()

print(f"[LANGCHAIN]")
print(f"Total pages loaded: {len(pages)}")
print(f"Page 1 Metadata: {pages[0].metadata}")
print(f"Page 1 text preview: \n{pages[0].page_content}\n")

/tmp/ipykernel_13138/4236737911.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[LANGCHAIN]
Total pages loaded: 40
Page 1 Metadata: {'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-17T00:01:43+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-17T00:01:43+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) kpathsea version 6.3.1', 'source': 'documents/sample.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1'}
Page 1 text preview: 
Journal of Machine Learning Research 26 (2025) 1-40 Submitted 10/23; Revised 11/24; Published 1/25
Locally Private Causal Inference for Randomized Experiments
Yuki Ohnishi yuki.ohnishi@yale.edu
Department of Biostatistics
Yale School of Public Health
New Haven, CT 06510, USA
Jordan Awan ja w an@purdue.edu
Department of Statistics
Purdue University
West Lafayette, IN 47907, USA
Editor: Elias Bareinboim
Abstract
Local diﬀerential privacy is a diﬀerential privacy paradigm in which individuals ﬁrst

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
# Configure the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800, # Maximum Character per chunk
    chunk_overlap = 150, # Sliding Window overlap between consecutive chunks
    length_function = len, # Measure chunk size by character count
    add_start_index=True, # Adds start_index to metadata (exact char position on the page)
)

In [4]:
# Split the pages Documents into smaller chunks Documents
chunks = text_splitter.split_documents(pages)
print(f"Total pages loaded: {len(pages)}")
print(f"Total chunks created: {len(chunks)}\n")

Total pages loaded: 40
Total chunks created: 158



In [13]:
# Inspect Chunk 0 and Chunk 1 to see the sliding window overlap in action
for i in range(3):
    print(f"=== CHUNK {i} ===\n")
    print(f"Metadata: {chunks[i].metadata}\n")
    print(f"Length: {len(chunks[i].page_content)} characters\n")
    print(f"End of Chunk {i} sentences... :\n{chunks[i].page_content}\n\n\n\n")

=== CHUNK 0 ===

Metadata: {'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-17T00:01:43+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-17T00:01:43+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) kpathsea version 6.3.1', 'source': 'documents/sample.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1', 'start_index': 0}

Length: 788 characters

End of Chunk 0 sentences... :
Journal of Machine Learning Research 26 (2025) 1-40 Submitted 10/23; Revised 11/24; Published 1/25
Locally Private Causal Inference for Randomized Experiments
Yuki Ohnishi yuki.ohnishi@yale.edu
Department of Biostatistics
Yale School of Public Health
New Haven, CT 06510, USA
Jordan Awan ja w an@purdue.edu
Department of Statistics
Purdue University
West Lafayette, IN 47907, USA
Editor: Elias Bareinboim
Abstract
Local diﬀerential privacy is a diﬀerential privacy paradigm

- Before letting ChromaDB do this behind the scenes in Step 4
let's write step3 embeddings to generate real embeddings with Ollama and calculate Cosine Similarity yourself on your first 3 PDF chunks:

In [10]:
import math
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

PDF_PATH = "documents/sample.pdf"

# 1. Helper function to compute Cosinge Similarity between two 768-float vectors
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    return dot_product / (norm_a * norm_b)

# 2. Initialize the local Ollama Embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

# 3. Quick sanity check: See sementics similarity on 3 test sentences
q_vec = embeddings.embed_query("How do neural networks learn from data?")
similar_vec = embeddings.embed_query("Deep learning models train by adjusting weights via backpropogation.")
unrelated_vec = embeddings.embed_query("The best recipe for crispy wood-fried pizza dough.")

print("=== PART A: Vector Shape & Semantic Math Check ===")
# print(q_vec)
print(f"Vector Dimension (Length): {len(q_vec)} floats")
print(f"First 5 numbers of query vector: {[round(x, 4) for x in q_vec[:5]]}")
print(f"Similarity (AI Question vs AI Sentence):    {cosine_similarity(q_vec, similar_vec):.4f}")
print(f"Similarity (AI Question vs Pizza Sentence): {cosine_similarity(q_vec, unrelated_vec):.4f}\n")

=== PART A: Vector Shape & Semantic Math Check ===
Vector Dimension (Length): 768 floats
First 5 numbers of query vector: [0.0228, 0.0134, -0.1423, -0.0082, 0.0944]
Similarity (AI Question vs AI Sentence):    0.6464
Similarity (AI Question vs Pizza Sentence): 0.2899



#### Load and chunk your PDF (Steps 1 & 2)

In [11]:
pages = PyPDFLoader(PDF_PATH).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, 
    chunk_overlap=150, 
    add_start_index=True
)
chunks = splitter.split_documents(pages)

#### Embed the first 3 chunks from your PDF using embed_documents()

In [12]:
first_3_texts = [chunk.page_content for chunk in chunks[:3]]
chunk_vectors = embeddings.embed_documents(first_3_texts)

print(first_3_texts)

['Journal of Machine Learning Research 26 (2025) 1-40 Submitted 10/23; Revised 11/24; Published 1/25\nLocally Private Causal Inference for Randomized Experiments\nYuki Ohnishi yuki.ohnishi@yale.edu\nDepartment of Biostatistics\nYale School of Public Health\nNew Haven, CT 06510, USA\nJordan Awan ja w an@purdue.edu\nDepartment of Statistics\nPurdue University\nWest Lafayette, IN 47907, USA\nEditor: Elias Bareinboim\nAbstract\nLocal diﬀerential privacy is a diﬀerential privacy paradigm in which individuals ﬁrst apply a\nprivacy mechanism to their data (often by adding noise) before transmitting the result to a\ncurator. The noise for privacy results in additional bias and variance in their analyses. Thus\nit is of great importance for analysts to incorporate the privacy noise into valid inference.', 'it is of great importance for analysts to incorporate the privacy noise into valid inference.\nIn this article, we develop methodologies to infer causal eﬀects from locally privatized\ndata u

In [16]:
# naïve debiased estimator results

# 6. Rank the first 3 chunks against a sample question about your PDF
sample_question = "What is the main topic or objective of this document?"
question_vec = embeddings.embed_query(sample_question)

In [17]:
print(f"=== PART B: Ranking First 3 PDF Chunks for: '{sample_question}' ===")
for i, (chunk, c_vec) in enumerate(zip(chunks[:3], chunk_vectors)):
    score = cosine_similarity(question_vec, c_vec)
    print(f"Chunk {i} (Page {chunk.metadata.get('page', 0) + 1}) -> Cosine Similarity: {score:.4f}")
    print(f"Preview: {chunk.page_content[:120]}...\n")

=== PART B: Ranking First 3 PDF Chunks for: 'What is the main topic or objective of this document?' ===
Chunk 0 (Page 1) -> Cosine Similarity: 0.3897
Preview: Journal of Machine Learning Research 26 (2025) 1-40 Submitted 10/23; Revised 11/24; Published 1/25
Locally Private Causa...

Chunk 1 (Page 1) -> Cosine Similarity: 0.3259
Preview: it is of great importance for analysts to incorporate the privacy noise into valid inference.
In this article, we develo...

Chunk 2 (Page 1) -> Cosine Similarity: 0.4214
Preview: applied to any of our proposed privacy mechanisms, and which performs especially well in
terms of MSE for tight privacy ...



In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

PDF_PATH = "documents/sample.pdf"
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "pdf_rag_langchain"

# 1. load and chunck the PDF (Steps 1 and 2)
pages = PyPDFLoader(PDF_PATH).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 150,
    add_start_index = True
)

chunks = splitter.split_documents(pages)

# 2. Initialize the local Ollama Embedding model (Step 3)
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

# 3. Create detereministics ID's for each chunk so re-running doesn't duplicate the data
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

# 4. Build (or update) the presistence ChromaDB vector store on disk
print(f"Indexing {len(chunks)} chunks into ChromaDB at {CHROMA_PATH}...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=chunk_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH
)
print(f"Total chunks stored in collection: {vector_store._collection.count()}\n")

# 5. Test Retrtival with similarity search with score (Top k=3)
query = "What are the main concepts or key findings discussed in this document?"
results = vector_store.similarity_search_with_score(query, k=3)

print(f"=== TOP 3 RETRIEVED CHUNKS FOR: '{query}' ===")
for rank, (doc, distance_score) in enumerate(results):
    page_num = doc.metadata.get("page", 0) + 1 # Convert 0-index page to 1-index for display
    start_idx = doc.metadata.get("start_index", "?")
    print(f"\n[Rank {rank}] Page {page_num} (char offset {start_idx}) | Distance Score: {distance_score:.4f} (lower is closer)")
    print(f"Text: {doc.page_content[:250]}...")

Indexing 158 chunks into ChromaDB at ./chroma_db...
Total chunks stored in collection: 158

=== TOP 3 RETRIEVED CHUNKS FOR: 'What are the main concepts or key findings discussed in this document?' ===

[Rank 0] Page 16 (char offset 0) | Distance Score: 0.9019 (lower is closer)
Text: Yuki Ohnishi and Jordan A w an
Table 3: Empirical analysis evaluating privatized cash transfer programs in Colombia. In the “Non-private” columns, “Freq”
represents the standard IPW estimator, while “Bayes” represents the standard Dirichlet process m...

[Rank 1] Page 18 (char offset 0) | Distance Score: 0.9785 (lower is closer)
Text: Yuki Ohnishi and Jordan A w an
covariate balances between treatment groups, which is particularly challenging in DP contexts.
Therefore, another future direction for research is to develop methodologies for PATE
estimation that address selection bias...

[Rank 2] Page 38 (char offset 2001) | Distance Score: 0.9810 (lower is closer)
Text: Annals of Applied Statistics, 2(1):176 

In [3]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "pdf_rag_langchain"

# 1. Reconnect to the existing ChromaDB on disk (using the exact same embedding model)
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings
)

# 2. Initialize the local Ollama Chat LLM with temperature=0 for factual accuracy 
llm = ChatOllama(
    model="llama3.2:latest"
)

# 3. Create a Augmentation Prompt Template to (System Rules + Context + Question)
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a factual research assistant. Answer the user's question using ONLY the provided context chunks.\n"
        "Always cite the page number in brackets like [Page X] when stating facts from a chunk.\n"
        "If the answer is not contained in the context, say: 'I cannot find the answer in the provided PDF.'",
    ),
    (
        "human",
        "Context Chunks:\n{context}\n\nQuestion: {question}",
    )
])

# 4. Connect the Prompt and LLM into a Langchain Pipeline using "|" pipe operator 
rag_chain = prompt_template | llm

def ask_pdf(question:str, k: int = 3):
    # Step A: Retrive top-k chunks + distance scores from ChromaDB
    retrieved_results = vector_store.similarity_search_with_score(question, k=k)
    
    # Step B: Format the chunks into a single string with 1-indexed Page tags
    formatted_chunks = []
    for i, (doc, score) in enumerate(retrieved_results):
        page_num = doc.metadata.get("page", 0) + 1
        formatted_chunks.append(f"--- Chunk {i} [Page {page_num}] ---\n{doc.page_content}")
    
    context_text = "\n\n".join(formatted_chunks)
    
    # Step C: Invoke the prompt -> LLM Chain
    response = rag_chain.invoke({
        "context": context_text,
        "question": question
    })
    
    # Step D: Print the LLM Answer and the Retrieved Chunks side-by-side (Stretch Goal)
    print("\n" + "=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)
    print("\n[LLM ANSWER]:")
    print(response.content)

    print("\n" + "-" * 60)
    print(f"[RETRIEVED SOURCE CHUNKS (Top {k})]:")
    print("-" * 60)
    for i, (doc, distance_score) in enumerate(retrieved_results, start=1):
        page_num = doc.metadata.get("page", 0) + 1
        start_idx = doc.metadata.get("start_index", "?")
        print(f"\nChunk #{i} | Page {page_num} (char offset {start_idx}) | Distance: {distance_score:.4f}")
        print(f"\"{doc.page_content.strip()}\"")
    print("=" * 60 + "\n")

# 5. Interactive CLI Chat Loop

if __name__ == "__main__":
    print(f"Connected to ChromaDB ({vector_store._collection.count()} chunks loaded).")
    print("Type a question about your PDF (or type 'exit' to quit):\n")
    while True:
        user_q = input("Ask PDF > ").strip()
        if user_q.lower() in {"exit", "quit", "q"}:
            break
        if user_q:
            ask_pdf(user_q, k=3)

Connected to ChromaDB (158 chunks loaded).
Type a question about your PDF (or type 'exit' to quit):


QUESTION: What is Rubin Causal Model?

[LLM ANSWER]:
According to Chunk 1 [Page 4], the Rubin Causal Model (RCM) is a causal paradigm adopted in this manuscript. Additionally, Chunk 2 [Page 1] doesn't explicitly define Rubin Causal Model, but Chunk 1 [Page 4] mentions that it is critical to define the Science of a particular problem, which includes defining the experimental units, covariates, treatments, and potential outcomes (Imbens and Rubin, 2015).

------------------------------------------------------------
[RETRIEVED SOURCE CHUNKS (Top 3)]:
------------------------------------------------------------

Chunk #1 | Page 4 (char offset 0) | Distance: 0.4622
"Yuki Ohnishi and Jordan A w an
of all variables, including treatment, outcomes, and covariates. Additionally, we emphasize
the context of randomized experiments, which remains the benchmark for credible causal
inference. Other k